# Predicting cuisine from ingredients with BERT

This is the What's Cooking dataset: given the ingredient list of a recipe, predict which cuisine it is. I fine-tuned bert-base-uncased with a small MLP head on top. Got 0.797 validation accuracy after 6 epochs.

In [3]:
import pandas as pd

In [4]:
test_path = "/kaggle/input/datasets/nikolozkiladze/training/test_whats_cooking.json"
train_path = "/kaggle/input/datasets/nikolozkiladze/training/train_whats_cooking.json"

In [5]:
train_pd = pd.read_json(train_path)
test_pd = pd.read_json(test_path)

In [6]:
train_list = []
max_len = 0
max_sen = ''
for i in range(39774):
  loc_text = train_pd.loc[:, 'ingredients'][i]
  ret = ''
  for k in range(len(loc_text)):
    ret += loc_text[k] + ' '
  max_len = max(max_len, len(ret))
  if max_len == len(ret):
    max_sen = ret
  train_list.append(ret)

#||TOKENIZE||

In [7]:
from transformers import BertTokenizerFast
from tqdm import tqdm
import torch
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
BATCH_SIZE = 1024  # Large batches work great for the CPU-bound fast tokenizer
MAX_LENGTH = 64
total_samples = len(train_list)

input_ids = []
attention_masks = []

print("Starting batch tokenization loop...")

for i in tqdm(range(0, total_samples, BATCH_SIZE), desc="Tokenizing Progress"):
    # Slice the raw strings
    batch_text = train_list[i : i + BATCH_SIZE]

    batch_encodings = tokenizer(
        batch_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'  # Return PyTorch tensors directly
    )

    # Store the tensors
    input_ids.append(batch_encodings['input_ids'])
    attention_masks.append(batch_encodings['attention_mask'])

# Concatenate all chunks into single giant matrices on the CPU
print("Finalizing matrices...")
final_input_ids = torch.cat(input_ids, dim=0)
final_attention_masks = torch.cat(attention_masks, dim=0)

# Package them nicely
full_encodings = {
    'input_ids': final_input_ids,
    'attention_mask': final_attention_masks
}

print("Done! Final tensor shape:", full_encodings['input_ids'].shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Starting batch tokenization loop...


Tokenizing Progress: 100%|██████████| 39/39 [00:04<00:00,  8.18it/s]

Finalizing matrices...
Done! Final tensor shape: torch.Size([39774, 64])


In [8]:
import torch
from transformers import BertModel
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

# =====================================================================
# 1. MODEL ARCHITECTURE
# =====================================================================
class BertFoodClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        # Clean bottleneck strategy
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.Dropout(p=0.2),
            nn.GELU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        # Full computational graph is preserved here for backpropagation
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_token)
        return logits

# Instantiate and push to GPU
model = BertFoodClassifier(num_classes=20).to('cuda')

# =====================================================================
# 2. OPTIMIZER & LOSS FUNCTION SETUP
# =====================================================================
criterion = nn.CrossEntropyLoss()

# Differential learning rates prevent destroying pre-trained BERT weights
optimizer = torch.optim.AdamW([
    {'params': model.bert.parameters(), 'lr': 2e-5},       # Slow fine-tuning for backbone
    {'params': model.classifier.parameters(), 'lr': 1e-3}  # Standard speed for your custom head
], weight_decay=0.01)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
len(full_encodings['input_ids'])

39774

In [56]:
lib = {}
lib_rev = {}
for idx, cus in enumerate(train_pd['cuisine'].unique()):
  lib[cus] = idx
  lib_rev[idx] = cus

In [14]:
import torch

# Map your dataframe column to integers directly
y = [lib[c] for c in train_pd['cuisine']]

# Convert the list to a PyTorch tensor
y = torch.tensor(y, dtype=torch.long)

In [15]:
from sklearn.model_selection import train_test_split

BATCH_SIZE = 32

input_ids = full_encodings['input_ids']
attention_mask = full_encodings['attention_mask']

# 3. Unpack them all together!
# The order of outputs matches the order of inputs: (ids_tr, ids_val, mask_tr, mask_val, y_tr, y_val)
(
    train_ids, val_ids,
    train_mask, val_mask,
    train_labels, val_labels
) = train_test_split(
    input_ids,
    attention_mask,
    y,
    test_size=0.15,
    random_state=42,   # Ensures reproducibility
    stratify=y  # Highly recommended: keeps class distributions balanced
)

# 4. Package them straight into your optimized TensorDatasets
train_dataset = TensorDataset(
    train_ids,
    train_mask,
    train_labels
)

val_dataset = TensorDataset(
    val_ids,
    val_mask,
    val_labels
)

# Optimized DataLoaders with pinned memory for lightning-fast host-to-device transfers
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

#Training MLP

In [16]:
epochs = 6
device = 'cuda'
train_acc_history = []
val_acc_history = []

print(f"Starting fine-tuning on {device}...")

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    epoch_train_correct = 0
    epoch_train_total = 0
    running_loss = 0.0

    # Wrapped in tqdm progress bar for crisp visibility
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for batch_input_ids, batch_attention_mask, batch_labels in train_loop:

        # Fast non-blocking host-to-device transfers
        X_ids = batch_input_ids.to(device, non_blocking=True)
        X_mask = batch_attention_mask.to(device, non_blocking=True)
        y_true = batch_labels.to(device, non_blocking=True)

        # Forward pass
        y_pred = model(input_ids=X_ids, attention_mask=X_mask)
        loss = criterion(y_pred, y_true)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track training statistics cleanly in the background
        with torch.no_grad():
            predictions_tr = y_pred.argmax(dim=1)
            epoch_train_correct += (predictions_tr == y_true).sum().item()
            epoch_train_total += y_true.size(0)
            running_loss += loss.item() * y_true.size(0)

            # Dynamic progress bar metrics
            train_loop.set_postfix(loss=loss.item())

    epoch_train_acc = 100 * epoch_train_correct / epoch_train_total
    train_acc_history.append(epoch_train_acc)
    average_train_loss = running_loss / epoch_train_total

    # --- EVALUATION PHASE ---
    model.eval()
    epoch_val_correct = 0
    epoch_val_total = 0

    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
    with torch.no_grad():
        for batch_input_ids, batch_attention_mask, batch_labels in val_loop:
            X_ids = batch_input_ids.to(device, non_blocking=True)
            X_mask = batch_attention_mask.to(device, non_blocking=True)
            y_true = batch_labels.to(device, non_blocking=True)

            outs = model(input_ids=X_ids, attention_mask=X_mask)
            predictions = torch.argmax(outs, dim=1)

            epoch_val_correct += (predictions == y_true).sum().item()
            epoch_val_total += y_true.size(0)

    epoch_val_acc = 100 * epoch_val_correct / epoch_val_total
    val_acc_history.append(epoch_val_acc)

    # Clean, scannable telemetry printout per epoch
    print(f"--> Epoch {epoch+1} Complete | Train Loss: {average_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | Val Acc: {epoch_val_acc:.2f}%\n")

Starting fine-tuning on cuda...


Epoch 1/6 [Train]: 100%|██████████| 1057/1057 [05:50<00:00,  3.02it/s, loss=1.22] 


--> Epoch 1 Complete | Train Loss: 1.0202 | Train Acc: 69.67% | Val Acc: 75.58%



Epoch 2/6 [Train]: 100%|██████████| 1057/1057 [05:54<00:00,  2.98it/s, loss=0.477]


--> Epoch 2 Complete | Train Loss: 0.7658 | Train Acc: 76.97% | Val Acc: 78.10%



Epoch 3/6 [Train]: 100%|██████████| 1057/1057 [05:54<00:00,  2.98it/s, loss=0.59] 


... (3 outputs left out)

Epoch 5/6 [Train]: 100%|██████████| 1057/1057 [05:54<00:00,  2.98it/s, loss=0.239] 


--> Epoch 5 Complete | Train Loss: 0.4492 | Train Acc: 86.11% | Val Acc: 79.50%



Epoch 6/6 [Train]: 100%|██████████| 1057/1057 [05:54<00:00,  2.98it/s, loss=0.28]  
                                                                  

--> Epoch 6 Complete | Train Loss: 0.3680 | Train Acc: 88.39% | Val Acc: 79.69%



In [32]:
test_list = []
max_len_tst = 0
max_sen_tst = ''
for i in range(9944):
  loc_text = test_pd.loc[:, 'ingredients'][i]
  ret = ''
  for k in range(len(loc_text)):
    ret += loc_text[k] + ' '
  max_len_tst = max(max_len, len(ret))
  if max_len_tst == len(ret):
    max_sen_tst = ret
  test_list.append(ret)

In [34]:
BATCH_SIZE = 1024  # Large batches work great for the CPU-bound fast tokenizer
MAX_LENGTH = 64
total_samples = len(test_list)

input_ids = []
attention_masks = []

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

for i in tqdm(range(0, total_samples, BATCH_SIZE), desc="Tokenizing Progress"):
    # Slice the raw strings
    batch_text = test_list[i : i + BATCH_SIZE]

    batch_encodings = tokenizer(
        batch_text,
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'  # Return PyTorch tensors directly
    )

    # Store the tensors
    input_ids.append(batch_encodings['input_ids'])
    attention_masks.append(batch_encodings['attention_mask'])

# Concatenate all chunks into single giant matrices on the CPU
print("Finalizing matrices...")
final_input_ids = torch.cat(input_ids, dim=0)
final_attention_masks = torch.cat(attention_masks, dim=0)

# Package them nicely
full_encodings_test = {
    'input_ids': final_input_ids,
    'attention_mask': final_attention_masks
}

print("Done! Final tensor shape:", full_encodings_test['input_ids'].shape)

Tokenizing Progress: 100%|██████████| 10/10 [00:01<00:00,  9.11it/s]

Finalizing matrices...
Done! Final tensor shape: torch.Size([9944, 64])


In [35]:
BATCH_SIZE = 32

test_input_ids = full_encodings_test['input_ids']
test_attention_mask = full_encodings_test['attention_mask']

# 3. Unpack them all together!
# The order of outputs matches the order of inputs: (ids_tr, ids_val, mask_tr, mask_val, y_tr, y_val)

# 4. Package them straight into your optimized TensorDatasets
test_dataset = TensorDataset(
    test_input_ids,
    test_attention_mask
)

# Optimized DataLoaders with pinned memory for lightning-fast host-to-device transfers
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, pin_memory=True)

In [38]:
preds = []
test_loop = tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
with torch.no_grad():
    for batch_input_ids, batch_attention_mask in test_loop:
        X_ids = batch_input_ids.to(device, non_blocking=True)
        X_mask = batch_attention_mask.to(device, non_blocking=True)
        y_true = batch_labels.to(device, non_blocking=True)

        outs = model(input_ids=X_ids, attention_mask=X_mask)
        predictions = torch.argmax(outs, dim=1)
        preds.append(predictions)

In [46]:
alls = torch.cat(preds, dim=0)

In [87]:
ans = []
for i in range(9944):
    ans.append(lib_rev[alls[i].cpu().item()])

In [86]:
len(test_pd.loc[:, 'id'].to_numpy())

9944

In [88]:
returner = pd.DataFrame({'id':test_pd.loc[:, 'id'].to_numpy(), 'cuisine':ans})

In [90]:
returner.to_csv("whose_cooking.csv")